# This is an Online Trained Logistic Regression Model on The US Accidents Data Set

There are many tries here, involving sklearn's native SGDClassifier
and, the river library for Online Learning

**The Notebook is a work in progress and part of the US Accidents Analysis Project**

**Link - https://www.kaggle.com/work/collections/18305074**

*Please Note, the Warnings have been specifically left on; so that the reader is aware of any poential changes or behaviour specifications.*

**This is the consensus of multiple training sessions**

The answers will be reported as the best discovered imbalanced classification report macro averages, 
but I looked at the full class wise report before picking the best model.

Do Note, the best discovered model has been retrained at the end; 
and the latest versions of the notebook will continue to report the same only. 
All the testing had been done privatelty.

Anyone interested in the model or any other steps if free to fork the notebook and try stuff out.
    
**Logistic Regression** performerd reasonably okay, the best set of metrics I was able to reach is -
    
                      pre       rec       spe        f1       geo       iba       sup
    
              1      0.000     0.000     1.000     0.000     0.000     0.000     20220
              2      0.813     0.925     0.166     0.865     0.391     0.165   1846275
              3      0.314     0.126     0.944     0.180     0.345     0.109    390162
              4      0.052     0.051     0.975     0.052     0.223     0.045     61862
    avg / total      0.701     0.759     0.326     0.721     0.376     0.151   2318519

Achieved by - penalty='elasticnet', l1_ratio=0.5, alpha=0.165

In [1]:
!git clone --filter=blob:none --no-checkout "https://github.com/Paras-GaurLRN/US-Accidents-EDA-Ensemble-Models.git"
%cd "US-Accidents-EDA-Ensemble-Models"
!git sparse-checkout init --cone
!git sparse-checkout set "notebooks/US Accidents - Pipelines/"
!git checkout main
%cd ..

Cloning into 'US-Accidents-EDA-Ensemble-Models'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 95 (delta 52), reused 65 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 11.33 KiB | 3.78 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/kaggle/working/US-Accidents-EDA-Ensemble-Models
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 17.92 MiB | 17.27 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (12/12), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
/kaggle/working


In [2]:
print("Files In The Repo-\n")

data_path = '/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines'

import os
for _, _, filenames in os.walk(data_path):
    for filename in filenames:
        print(filename)

Files In The Repo-

us-accidents-pipeline.ipynb
Libraries.txt
pipeline.py
Imports.txt


In [3]:
print("The Dataset-\n")

dataset_path = '/kaggle/input'

for _, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(filename)

The Dataset-

US_Accidents_March23.csv
__results__.html
__notebook__.ipynb
__output__.json
custom.css


In [4]:
import sys

sys.path.append(data_path)

# To ensure that we can import the pipe

In [5]:
print("### Libraires ###\n")
with open(f'{data_path}/Libraries.txt') as LibrariesTXT:
    for line in LibrariesTXT.readlines():
        print(line)

### Libraires ###

scikit-learn

imbalanced-learn

feature-engine


In [6]:
print("### Imports ###\n")
with open(f'{data_path}/Imports.txt') as ImportsTXT:
    for line in ImportsTXT.readlines():
        print(line)

### Imports ###

from warnings import warn

from sklearn.base import (BaseEstimator, TransformerMixin, clone)

from sklearn.utils._param_validation import StrOptions

from imblearn.base import BaseSampler

from sklearn.utils.validation import check_is_fitted

from sklearn.compose import ColumnTransformer

from feature_engine.datetime import DatetimeFeatures

from feature_engine.outliers import ArbitraryOutlierCapper

from imblearn.pipeline import Pipeline as IMBPipe

import pandas as pd

import numpy as np


In [7]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 4.4 MB/s eta 0:00:00


In [8]:
!pip list | grep -E "numpy|pandas|feature-engine"

geopandas                                1.1.3
numpy                                    2.4.6
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
sklearn-pandas                           2.2.0


**Note: Target column = Severity**

# SGDClassifier

In [9]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import (StandardScaler, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [10]:
from pipeline import (AnomalyCleaner, DateTimeFeatureEngineer, ColumnDropper, Illuminator)

**We Need To Discuss all the decisions made in the following code here,**

**1)** Firstly, the loop is made accordingly due to different transformers behaving differently.

We have 3 clasifications-

*SL* : Stateless = These do not store anything that would need to be propogated across the loop.

*SI* : Stateful, but Incremental = These require propogation of learnt values across fits, but are build for Incremental Learning.

*SN* : Stateful, non Incremental = These are Stateful but not built for Incremental Learning.

*SL* & *SI* are easy to work with, but *SN* require inspection in each loop. We also need to ensure no **Evaluation Leakage** whatsoever.

**2)** The First iteration is the learning and logging stage; it requires the most attention, rest stages are simple.

**3)** All iterations forward need to account for the State of a transformer, past values & appropriate transformations.

**The Column Map**

*Pre-categories encoded for simplicity, OHE makes the final result a bit larger*
![Column Map](https://i.postimg.cc/zvyyYypX/Model-1.png)

In [11]:
from sklearn.base import (BaseEstimator, TransformerMixin)
from sklearn.utils.validation import check_is_fitted

# Specifically designed to be SL, finer-details of the implementation may change in some other
# Out-Of-Core Project as it depends on our aim.
# Mean is more streamable and thus used, although we can even use median; but
# Then batch based loading will not work.

class OutOfCoreNumericalImputer(BaseEstimator, TransformerMixin):
    def __init__(self,*,
                 file_name,
                 columns,
                 batch_size = None,
                 copy = False):
        self.file_name = file_name
        self.columns = columns
        self.copy = copy
        self.batch_size = batch_size
    
    def fit(self, X=None, y=None):
        means = {}
        for col in self.columns:
            if self.batch_size is None:
                data = pd.read_csv(self.file_name,index_col='ID',usecols=['ID',col])
                means[col] = (data[col].sum(skipna=True))/(data[col].notna().sum())
            else:
                _sum = 0
                _count = 0
                for chunk in pd.read_csv(self.file_name,index_col='ID',usecols=['ID',col],chunksize=self.batch_size):
                    _sum += chunk[col].sum(skipna=True)
                    _count += chunk.notna().sum()
                means[col] = _sum/_count

        self._means = means
        
        return self
        
    def transform(self, chunk):
        check_is_fitted(self, "_means")

        if sorted((chunk.columns).to_list()) != sorted(self.columns): raise ValueError('Transformation columns do not match fitted columns')
        
        if self.copy: chunk = chunk.copy()

        for col in self.columns:
            chunk[col] = chunk[col].fillna(self._means[col])
        
        return chunk

In [12]:
from sklearn import set_config
set_config(transform_output='pandas')

Train-Test Spliting The Data First

In [13]:
from sklearn.model_selection import train_test_split

FILE = '/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv'

TRAIN_FILE = 'US_Accidents_train.csv'
TEST_FILE = 'US_Accidents_test.csv'

first_train = True
first_test = True

for chunk in pd.read_csv(
    FILE,
    index_col='ID',
    chunksize=20_000
):
    train_chunk, test_chunk = train_test_split(
        chunk,
        test_size=0.30,
        random_state=34
    )

    train_chunk.to_csv(
        TRAIN_FILE,
        mode='w' if first_train else 'a',
        header=first_train,
        index=True
    )

    test_chunk.to_csv(
        TEST_FILE,
        mode='w' if first_test else 'a',
        header=first_test,
        index=True
    )

    first_train = False
    first_test = False

    del train_chunk, test_chunk, chunk

print("Data Split Created!")

Data Split Created!


First Pass For Transformer Training

In [14]:
drop_cols = ["Amenity","Bump","Crossing","Give_Way","Junction","No_Exit","Railway","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal"]
rem_num_cols = ["Temperature(F)","Humidity(%)","Pressure(in)","Visibility(mi)","Wind_Speed(mph)","Precipitation(in)"]
rem_cat_cols = ["Source","Wind_Direction","Weather_Condition","Illumination"]
rem_datetime_cols = ["Accident Day","Accident Timing","Accident Year","Weekend"]
first_iter = True

T1_AC = AnomalyCleaner() # SL
T2_DTFE = DateTimeFeatureEngineer() # SL
T3_IL = Illuminator() # SL
T4_CD = ColumnDropper(columns= drop_cols + [col for col in ColumnDropper.DEFAULT_COLUMNS if col != 'Precipitation(in)']) # SL
T5_SIM_CAT = SimpleImputer(strategy='constant',fill_value='Missing',copy=False) # SL, categorical imputer is SL but a numerical analog will be SN
T5_SIM_NUM = OutOfCoreNumericalImputer(file_name = '/kaggle/working/US_Accidents_train.csv',
                                       columns = rem_num_cols) # SL

T5_SIM = ColumnTransformer([
    ('num',T5_SIM_NUM,rem_num_cols),
    ('cat',T5_SIM_CAT,[c for c in rem_cat_cols if c != 'Source'])
],remainder='passthrough',n_jobs=-1,verbose=False,verbose_feature_names_out=False) # SL, due to the Transformers used

T6_ENC_OE = OrdinalEncoder(categories=[['Missing',*(val for key,val in T3_IL.DEFAULT_ILLUMINATION_ORDER.items() if key != 'Miscellaneous')]],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1,
                           encoded_missing_value=-1) # SI

T6_ENC_OHE = OneHotEncoder(categories='auto',drop='first',sparse_output=False,handle_unknown='infrequent_if_exist',min_frequency=1e-5) # SI

T6_ENC = ColumnTransformer([
    ('ohe',T6_ENC_OHE,['Wind_Direction','Weather_Condition']),
    ('ord',T6_ENC_OE,['Illumination'])
],remainder='passthrough',n_jobs=-1,verbose=False,verbose_feature_names_out=False) # SI, due to the transformers used

T7_SS = StandardScaler() # SI

for chunk in pd.read_csv('/kaggle/working/US_Accidents_train.csv',
                         index_col='ID',
                         chunksize=20000):
    if first_iter:
        X, y = chunk.drop(columns=['Severity']), chunk['Severity']
        
        Wind_Direction = (
            pd.read_csv(
                '/kaggle/working/US_Accidents_train.csv',
                usecols=['Wind_Direction']
            )['Wind_Direction']
        )

        Weather_Condition = (
            pd.read_csv(
                '/kaggle/working/US_Accidents_train.csv',
                usecols=['Weather_Condition']
            )['Weather_Condition']
        )
        
        T1_AC.fit(X)
        X, y = T1_AC.fit_resample(X, y)
        
        X = T2_DTFE.fit_transform(X)
        
        X = T3_IL.fit_transform(X)
        
        X = T4_CD.fit_transform(X)
        
        X = T5_SIM.fit_transform(X)
        
        X = T6_ENC.fit(pd.DataFrame({
            'Wind_Direction' : Wind_Direction.fillna('Missing'),
            'Weather_Condition' : Weather_Condition.fillna('Missing'),
            'Illumination' : np.full(shape=Wind_Direction.to_numpy().shape,fill_value='I')
        })).transform(X)

        del Weather_Condition
        del Wind_Direction
        
        T7_SS.partial_fit(X)
        
        first_iter = False
    else:
        X, y = chunk.drop(columns=['Severity']), chunk['Severity']

        X, y = T1_AC.fit_resample(X, y)
        X = T2_DTFE.transform(X)
        X = T3_IL.transform(X)
        X = T4_CD.transform(X)
        X = T5_SIM.transform(X)
        X = T6_ENC.transform(X)
        T7_SS.partial_fit(X)

print("Transformers Trained!")

/kaggle/working/US-Accidents-EDA-Ensemble-Models/notebooks/US Accidents - Pipelines/pipeline.py:97: UserWarning: AnomalyCleaner has a dual sampler/transformer interface and is customized for compatibility with imbalanced-learn's Pipeline API. When used inside an imblearn Pipeline, AnomalyCleaner is treated as a sampler: it participates during fitting via fit_resample(), but samplers are skipped during prediction/inference. Consequently, transform()-based capping of anomalous observations will not be applied automatically by pipeline.predict(). To apply inference-time capping, either use AnomalyCleaner.transform() independently before passing the data to the remaining pipeline steps, or treat validation and capping of prediction data as the responsibility of the calling API.
  warn(


Transformers Trained!


Sinking Learnt Transformers

In [15]:
import joblib

joblib.dump(
    {
        'T1_AC' : T1_AC,
        'T2_DTFE' : T2_DTFE,
        'T3_IL' : T3_IL,
        'T4_CD' : T4_CD,
        'T5_SIM' : T5_SIM,
        'T6_ENC' : T6_ENC,
        'T7_SS' : T7_SS
    },
    'Transformers.pkl'
)

print("Transformers Sinked!")

Transformers Sinked!


Second Pass For Model Training

In [16]:
classes = np.asarray([1,2,3,4])

y_counts = (
    pd.read_csv(TRAIN_FILE, usecols=["Severity"])["Severity"]
    .value_counts()
)

N = y_counts.sum()
K = len(y_counts)

class_weights = {
    cls: N / (K * count)
    for cls, count in y_counts.items()
}

In [17]:
SGDmodel = SGDClassifier(n_jobs=-1,verbose=0,shuffle=True,random_state=34,loss='log_loss',
                         penalty='elasticnet',
                         l1_ratio=0.5,
                         alpha=0.165)

first_iter = True
for chunk in pd.read_csv('/kaggle/working/US_Accidents_train.csv',
                         index_col='ID',
                         chunksize=20000):
    
    X, y = chunk.drop(columns=['Severity']), chunk['Severity']

    X, y = T1_AC.fit_resample(X, y)
    
    sample_weight = y.map(class_weights).to_numpy()
    
    X = T2_DTFE.transform(X)
    
    X = T3_IL.transform(X)
    
    X = T4_CD.transform(X)
    
    X = T5_SIM.transform(X)
    
    X = T6_ENC.transform(X)
    
    X = T7_SS.transform(X)

    if first_iter: SGDmodel.partial_fit(X, y, classes=classes, sample_weight=sample_weight); first_iter = False
    else: SGDmodel.partial_fit(X, y, sample_weight=sample_weight)

print("Model Trained!")

Model Trained!


Saving the SGDModel

In [18]:
joblib.dump(SGDmodel,'SGDmodel.pkl')

['SGDmodel.pkl']

In [19]:
from imblearn.metrics import classification_report_imbalanced

test_set = pd.read_csv('/kaggle/working/US_Accidents_test.csv',index_col='ID')

X_pred, y_true = test_set.drop(columns=['Severity']), test_set['Severity']
del test_set

X = T1_AC.transform(X_pred, issue_warning=False)

X = T2_DTFE.transform(X)

X = T3_IL.transform(X)

X = T4_CD.transform(X)

X = T5_SIM.transform(X)

X = T6_ENC.transform(X)

X = T7_SS.transform(X)

y_pred = SGDmodel.predict(X)

print("Classification Report : ")
print(classification_report_imbalanced(y_true, y_pred, digits=3))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Classification Report : 


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   pre       rec       spe        f1       geo       iba       sup

          1      0.000     0.000     1.000     0.000     0.000     0.000     20220
          2      0.813     0.925     0.166     0.865     0.391     0.165   1846275
          3      0.314     0.126     0.944     0.180     0.345     0.109    390162
          4      0.052     0.051     0.975     0.052     0.223     0.045     61862

avg / total      0.701     0.759     0.326     0.721     0.376     0.151   2318519



In [20]:
# # Uncomment if you wish to see the coefficients of the model

# feature_names = SGDmodel.feature_names_in_

# coef_df = pd.DataFrame(
#     SGDmodel.coef_.T,
#     index=feature_names,
#     columns=[f"Class_{c}" for c in SGDmodel.classes_]
# )
# with pd.option_context(
#     "display.max_rows", None,
#     "display.max_columns", None,
#     "display.width", None,
#     "display.expand_frame_repr", False
# ):
#     print(coef_df.sort_values("Class_4", ascending=False).to_string(max_rows=None,max_cols=None))